## Data preprocessing

### 1. Prepare input dataset for prediction model 

Filter out outlier from 'divemt_all_qe_scores.tsv'

In [1]:
import pandas as pd

In [2]:
file_path = '/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/data/dataset_qe/divemt_all_qe_scores.tsv'
model_df = pd.read_csv(file_path,sep ='\t')
#model_df.columns

In [3]:
"""
#quick check
model_columns = [
    "unit_id",
    "doc_id",
    "lang_id",
    "task_type",
    "mt_len_wrd",
    "src_len_wrd",
    "edit_time",
    "qe_cometkiwi",
    "qe_gemba_gemma3",
    "qe_gemba_gpt4_1_mini"
]
print (f'df shape: ',{model_df.shape})
print (f'Unique values in lang_id: ', model_df['lang_id'].unique())
print (f'Unique values in task_type: ',model_df['task_type'].unique())
print (model_df.groupby(['lang_id','task_type']).size())

for n, col in enumerate(model_columns):
    print (n, col, model_df[col].isna().sum())
"""



'\n#quick check\nmodel_columns = [\n    "unit_id",\n    "doc_id",\n    "lang_id",\n    "task_type",\n    "mt_len_wrd",\n    "src_len_wrd",\n    "edit_time",\n    "qe_cometkiwi",\n    "qe_gemba_gemma3",\n    "qe_gemba_gpt4_1_mini"\n]\nprint (f\'df shape: \',{model_df.shape})\nprint (f\'Unique values in lang_id: \', model_df[\'lang_id\'].unique())\nprint (f\'Unique values in task_type: \',model_df[\'task_type\'].unique())\nprint (model_df.groupby([\'lang_id\',\'task_type\']).size())\n\nfor n, col in enumerate(model_columns):\n    print (n, col, model_df[col].isna().sum())\n'

In [4]:
#inspect dependent variable
model_df['edit_time'].describe()

count     5160.000000
mean       110.561768
std        357.506522
min          0.359000
25%         32.100500
50%         62.140000
75%        115.390250
max      18094.487000
Name: edit_time, dtype: float64

In [5]:
model_df.groupby('lang_id')['edit_time'].describe()

,count,mean,std,min,25%,50%,75%,max
lang_id,,,,,,,,
ara,860.0,118.292984,354.069816,2.477,30.59500,57.1865,107.01050,8257.200
ita,860.0,88.875769,94.162217,0.985,30.61350,65.1775,115.67475,1380.862
nld,860.0,61.600501,96.234772,0.685,18.98450,42.6425,76.42850,2000.656
tur,860.0,64.546573,85.009380,0.359,23.65925,44.5075,78.63300,1650.252
ukr,860.0,137.105870,141.400191,1.079,56.99650,101.2095,168.61650,2412.522
vie,860.0,192.948909,764.424130,6.324,48.16375,82.2210,148.20650,18094.487


In [6]:
#Check for outlier in edit_time that exceed 45 mins (following divEMT)
outlier_threshold = 45 * 60

model_df_processed = model_df.loc[
    model_df["edit_time"] < outlier_threshold
].copy()
print("Before:", model_df.shape)
print("After:", model_df_processed.shape)
print("Removed:", len(model_df) - len(model_df_processed))
removed_outliers = model_df.loc[
    model_df["edit_time"] >= outlier_threshold
]

print(
    removed_outliers.groupby("lang_id").size()
)
model_df_processed["mt_system"] = (model_df_processed["task_type"] == "pe2").astype(int)
"""
model_df_processed.to_csv(
    "../data/processed/divemt_model_input.tsv",
    sep="\t",
    index=False
)
"""
model_df_processed.columns


Before: (5160, 69)
After: (5147, 69)
Removed: 13
lang_id
ara     2
vie    11
dtype: int64


Index(['unit_id', 'flores_id', 'item_id', 'subject_id', 'task_type',
       'translation_type', 'src_len_chr', 'mt_len_chr', 'tgt_len_chr',
       'src_len_wrd', 'mt_len_wrd', 'tgt_len_wrd', 'edit_time', 'k_total',
       'k_letter', 'k_digit', 'k_white', 'k_symbol', 'k_nav', 'k_erase',
       'k_copy', 'k_cut', 'k_paste', 'k_do', 'n_pause_geq_300',
       'len_pause_geq_300', 'n_pause_geq_1000', 'len_pause_geq_1000',
       'event_time', 'num_annotations', 'last_modification_time', 'n_insert',
       'n_delete', 'n_substitute', 'n_shift', 'tot_shifted_words', 'tot_edits',
       'hter', 'cer', 'bleu', 'chrf', 'lang_id', 'doc_id', 'time_s', 'time_m',
       'time_h', 'time_per_char', 'time_per_word', 'key_per_char',
       'words_per_hour', 'words_per_minute', 'per_subject_visit_order',
       'src_text', 'mt_text', 'tgt_text', 'aligned_edit', 'src_tokens',
       'src_annotations', 'mt_tokens', 'mt_annotations', 'tgt_tokens',
       'tgt_annotations', 'src_wmt22_qe', 'mt_wmt22_qe', 'q